# Estudo de Caso 10.7 — Análise Térmica de Aleta por Diferenças Finitas: Convergência e Validação

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 10 — Aletas e Superfícies Estendidas |
| **Nível** | Pós-Graduação |
| **Tempo estimado** | 90 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Diferenças Finitas — Sistema Tridiagonal |
| **Bibliografia** | Incropera et al. (2017), Cap. 3, Seções 3.6–3.7; Gobbi (s.d.), Cap. 2 |

---

## 📋 Resumo

Aplicamos o Método das Diferenças Finitas (MDF) para determinar o perfil de temperatura e a taxa de transferência de calor em uma aleta retangular de alumínio com ponta adiabática. A equação governante de segunda ordem é discretizada por diferenças centrais, gerando um sistema linear tridiagonal. A solução numérica é validada contra a solução analítica exata, e um estudo de independência de malha demonstra a convergência de segunda ordem ($\mathcal{O}(\Delta x^2)$) do esquema.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Discretizar** a equação diferencial da aleta por diferenças finitas centrais
2. **Montar** o sistema linear tridiagonal com condições de contorno de Dirichlet e Neumann
3. **Implementar** a solução numérica para malhas de diferentes resoluções
4. **Comparar** resultados numéricos com a solução analítica exata
5. **Analisar** a convergência de segunda ordem do esquema de diferenças centrais
6. **Calcular** o fluxo de calor na base da aleta pela Lei de Fourier discreta
7. **Avaliar** o compromisso entre precisão e custo computacional no refinamento de malha

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## ⚙️ Seção 1: Parâmetros da Aleta

| Parâmetro | Símbolo | Valor | Unidade |
|:----------|:-------:|------:|:-------:|
| Comprimento da aleta | $L$ | 0,10 | m |
| Largura | $w$ | 0,05 | m |
| Espessura | $t$ | 0,005 | m |
| Condutividade térmica (Al 2024) | $k$ | 177 | W/(m·K) |
| Coeficiente convectivo | $h$ | 25 | W/(m²·K) |
| Temperatura da base | $T_b$ | 150 | °C |
| Temperatura ambiente | $T_\infty$ | 25 | °C |

In [ ]:
# ============================================================
# PARÂMETROS DA ALETA
# ============================================================

# --- Geometria ---
L = 0.10              # Comprimento da aleta [m]
w = 0.05              # Largura [m]
t = 0.005             # Espessura [m]

# --- Propriedades térmicas (Alumínio 2024) ---
k = 177.0             # Condutividade térmica [W/(m·K)]
h_conv = 25.0         # Coeficiente de convecção [W/(m²·K)]

# --- Condições de contorno e ambiente ---
T_b = 150.0           # Temperatura da base [°C]
T_inf = 25.0          # Temperatura do ar ambiente [°C]

# --- Propriedades derivadas ---
A_c = w * t                        # Área da seção transversal [m²]
P = 2.0 * (w + t)                  # Perímetro da seção [m]
m = np.sqrt(h_conv * P / (k * A_c))  # Parâmetro da aleta [m⁻¹]

print('📏 Parâmetros da Aleta:')
print(f'  • Geometria: L={L*100:.0f} cm, w={w*100:.0f} cm, t={t*1000:.0f} mm')
print(f'  • Material: k={k} W/(m·K), h={h_conv} W/(m²·K)')
print(f'  • A_c = {A_c:.2e} m² | P = {P:.4f} m')
print(f'  • m = {m:.4f} m⁻¹ | mL = {m*L:.4f} (adimensional)')

---
## 🔬 Seção 2: Verificação — Solução Analítica de Referência

Antes da solução numérica, calculamos a solução analítica exata para servir como referência de validação.

Para uma aleta com ponta adiabática, a solução exata é:

$$\theta(x) = \theta_b \frac{\cosh[m(L-x)]}{\cosh(mL)}$$

onde $\theta(x) = T(x) - T_\infty$ e $\theta_b = T_b - T_\infty$.

O calor total dissipado é:

$$q_{\text{analítico}} = \sqrt{hPkA_c}\;\theta_b\;\tanh(mL)$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 3, Seção 3.6, Eq. 3.75.

In [ ]:
# ============================================================
# SOLUÇÃO ANALÍTICA (referência para validação)
# ============================================================

theta_b = T_b - T_inf  # Diferença de temperatura na base [K]

def T_analitico(x):
    """
    Perfil de temperatura analítico para aleta com ponta adiabática.

    Parameters
    ----------
    x : float ou ndarray
        Posição ao longo da aleta [m].

    Returns
    -------
    float ou ndarray
        Temperatura na posição x [°C].

    Notes
    -----
    Solução exata: T(x) = T_inf + theta_b * cosh(m(L-x)) / cosh(mL)
    """
    return T_inf + theta_b * np.cosh(m * (L - x)) / np.cosh(m * L)

# Calor dissipado (solução analítica)
q_analitico = np.sqrt(h_conv * P * k * A_c) * theta_b * np.tanh(m * L)

print('🔬 Solução Analítica de Referência:')
print(f'  • θ_b = {theta_b:.1f} K')
print(f'  • T(L) = {T_analitico(L):.2f} °C (temperatura na ponta)')
print(f'  • q_analítico = {q_analitico:.4f} W')
print(f'\n✅ Referência estabelecida para validação numérica')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Equação Governante

A equação diferencial para a aleta em regime permanente é:

$$\frac{d^2T}{dx^2} - m^2(T - T_\infty) = 0$$

onde:
- $T(x)$ = temperatura na posição $x$ [°C]
- $m = \sqrt{hP/(kA_c)}$ = parâmetro da aleta [m⁻¹]
- $x$ = coordenada longitudinal [m]

### 3.2 Discretização por Diferenças Centrais

A derivada segunda é aproximada por diferenças centrais de segunda ordem:

$$\frac{T_{i-1} - 2T_i + T_{i+1}}{(\Delta x)^2} - m^2(T_i - T_\infty) = 0$$

Reorganizando para os **nós internos** ($i = 1, 2, \ldots, N-2$):

$$T_{i-1} - \left[2 + m^2(\Delta x)^2\right] T_i + T_{i+1} = -m^2(\Delta x)^2\,T_\infty$$

### 3.3 Condições de Contorno

| Contorno | Tipo | Formulação |
|:---------|:-----|:-----------|
| Base ($x=0$, nó 0) | Dirichlet | $T_0 = T_b$ |
| Ponta ($x=L$, nó $N$-1) | Neumann (adiabática) | $\frac{dT}{dx}\big|_{x=L} = 0 \Rightarrow T_N = T_{N-2}$ (nó fictício) |

A condição de ponta adiabática no último nó resulta em:

$$2T_{N-2} - \left[2 + m^2(\Delta x)^2\right] T_{N-1} = -m^2(\Delta x)^2\,T_\infty$$

> 📖 **Referência:** Incropera et al. (2017), Cap. 3, Seção 3.6; Gobbi (s.d.), Cap. 2.

---
## ⚡ Seção 4: Critério Numérico e Estabilidade

> 💡 **Observação:** Neste problema de **regime permanente** com solução por sistema linear direto, não se aplica a condição CFL (que é relevante para problemas transientes explícitos). O critério de adequação é:

1. **Diagonal dominância:** a matriz do sistema deve ser diagonalmente dominante para garantir estabilidade do solver iterativo (quando aplicável). Para o solver direto (`np.linalg.solve`), a condição é que a matriz seja não-singular.

2. **Ordem de precisão:** o esquema de diferenças centrais é de segunda ordem, $\mathcal{O}(\Delta x^2)$, o que significa que ao reduzir $\Delta x$ pela metade, o erro deve cair por um fator de ~4.

3. **Número de nós mínimo:** para capturar adequadamente o perfil exponencial, recomenda-se $m\Delta x < 1$, ou seja:

$$N > m \cdot L + 1$$

In [ ]:
# ============================================================
# VERIFICAÇÃO DO CRITÉRIO NUMÉRICO
# ============================================================

# Número mínimo de nós para m*dx < 1
N_min_crit = int(np.ceil(m * L)) + 2

print('⚡ Critério Numérico:')
print(f'  • mL = {m*L:.4f}')
print(f'  • Número mínimo de nós (mΔx < 1): N ≥ {N_min_crit}')
print(f'  • Malha grossa: N=5 → mΔx = {m*L/4:.4f}')
print(f'  • Malha refinada: N=21 → mΔx = {m*L/20:.4f}')
print(f'\n✅ Ambas as malhas satisfazem mΔx < 1')

---
## 🌪️ Seção 5: Condições de Entrada (Forçamento Térmico)

Neste problema de regime permanente, o forçamento é a **temperatura prescrita na base** da aleta:

$$T(0) = T_b = 150 \text{ °C (Dirichlet)}$$

e a **condição de contorno convectiva** nas superfícies laterais, já incorporada no parâmetro $m$:

$$-k\frac{dT}{dx}\bigg|_{\text{superfície}} = h(T_s - T_\infty)$$

> 💡 **Observação:** A condição de ponta adiabática ($dT/dx = 0$ em $x=L$) é uma aproximação válida quando a área da ponta é muito menor que a área lateral, o que é o caso aqui ($A_{\text{ponta}}/A_{\text{lateral}} = t/(2L) \approx 0{,}025$).

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# FUNÇÃO PRINCIPAL: Resolução da aleta por Diferenças Finitas
# ============================================================

def resolver_aleta_mdf(N):
    """
    Resolve o perfil de temperatura da aleta por diferenças finitas.

    Parameters
    ----------
    N : int
        Número total de nós (incluindo base e ponta).

    Returns
    -------
    tuple
        (x, T, dx) onde:
        - x : ndarray, vetor de posições [m]
        - T : ndarray, vetor de temperaturas [°C]
        - dx : float, espaçamento entre nós [m]

    Notes
    -----
    Esquema de diferenças centrais de 2ª ordem.
    Condições de contorno: Dirichlet na base, Neumann (adiabática) na ponta.
    """
    dx = L / (N - 1)
    x = np.linspace(0, L, N)

    # Coeficiente da equação discretizada
    coef = 2.0 + (m * dx) ** 2
    fonte = -(m * dx) ** 2 * T_inf

    # Incógnitas: T_1, T_2, ..., T_{N-1}  (T_0 é conhecido)
    n_inc = N - 1
    A = np.zeros((n_inc, n_inc))
    b = np.zeros(n_inc)

    # Preenchimento para nós internos (i = 1 até N-2)
    for i in range(n_inc - 1):
        A[i, i] = -coef
        b[i] = fonte
        if i > 0:
            A[i, i - 1] = 1.0
        A[i, i + 1] = 1.0

    # Termo da base (T_0 = T_b) na primeira equação
    b[0] -= T_b

    # Última linha: condição de ponta adiabática
    # T_{N} = T_{N-2}  =>  2*T_{N-2} - coef*T_{N-1} = fonte
    A[-1, -1] = -coef
    A[-1, -2] = 2.0
    b[-1] = fonte

    # Resolução do sistema linear
    T_int = np.linalg.solve(A, b)

    # Monta o vetor completo de temperaturas
    T = np.concatenate(([T_b], T_int))

    return x, T, dx

In [ ]:
# ============================================================
# CASO BASE: Malha grossa (5 nós)
# ============================================================

t_inicio = time.time()
x5, T5, dx5 = resolver_aleta_mdf(5)
t_cpu_5 = time.time() - t_inicio

print('=' * 60)
print('  MALHA GROSSA: 5 nós (Δx = {:.1f} mm)'.format(dx5 * 1000))
print('=' * 60)
print(f'\n{"x [m]":<10}{"T_MDF [°C]":<15}{"T_analítico [°C]":<18}{"Erro [%]":<10}')
print('-' * 55)
for xi, Ti in zip(x5, T5):
    Ta = T_analitico(xi)
    erro = 100.0 * abs(Ti - Ta) / abs(Ta - T_inf)
    print(f'{xi:<10.3f}{Ti:<15.2f}{Ta:<18.2f}{erro:<10.3f}')

print(f'\n⏱️ Tempo de CPU: {t_cpu_5:.6f} s')

In [ ]:
# ============================================================
# MALHA REFINADA: 21 nós
# ============================================================

t_inicio = time.time()
x21, T21, dx21 = resolver_aleta_mdf(21)
t_cpu_21 = time.time() - t_inicio

print('=' * 60)
print('  MALHA REFINADA: 21 nós (Δx = {:.1f} mm)'.format(dx21 * 1000))
print('=' * 60)
print(f'\n{"x [m]":<10}{"T_MDF [°C]":<15}{"T_analítico [°C]":<18}{"Erro [%]":<10}')
print('-' * 55)
# Comparar nos mesmos pontos da malha grossa
for xi in x5:
    T21_interp = np.interp(xi, x21, T21)
    Ta = T_analitico(xi)
    erro = 100.0 * abs(T21_interp - Ta) / abs(Ta - T_inf)
    print(f'{xi:<10.3f}{T21_interp:<15.2f}{Ta:<18.2f}{erro:<10.3f}')

print(f'\n⏱️ Tempo de CPU: {t_cpu_21:.6f} s')

> 💡 **Observação:**
> - A malha grossa (5 nós) já fornece boa estimativa de temperatura, mas com erro significativo no gradiente
> - A malha refinada (21 nós) reduz o erro para valores abaixo de 0,1%
> - O erro é máximo na região de maior curvatura do perfil (próximo à base)

---
## 📊 Seção 7: Visualização dos Perfis de Temperatura

In [ ]:
# ============================================================
# GRÁFICO: Perfis de temperatura (analítico vs numérico)
# ============================================================

# Curva analítica de alta resolução
x_ana = np.linspace(0, L, 200)
T_ana = T_analitico(x_ana)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(x_ana * 100, T_ana, 'k-', linewidth=2.5, label='Solução analítica')
ax.plot(x5 * 100, T5, 'ro-', markersize=9, linewidth=1.5,
        label='MDF — 5 nós (Δx = 25 mm)')
ax.plot(x21 * 100, T21, 'bs-', markersize=5, linewidth=1.2,
        label='MDF — 21 nós (Δx = 5 mm)')

ax.set_xlabel('Posição ao longo da aleta x [cm]', fontsize=12)
ax.set_ylabel('Temperatura T [°C]', fontsize=12)
ax.set_title('Perfil de Temperatura na Aleta — Analítico × Numérico',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, L * 100)
ax.set_ylim(T_inf, T_b + 10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - O perfil de temperatura decai exponencialmente da base (150 °C) em direção à ponta
> - Com 5 nós, a discretização é grosseira mas já captura a tendência geral
> - Com 21 nós, os pontos numéricos são praticamente indistinguíveis da curva analítica
> - A condição de ponta adiabática é evidenciada pelo gradiente nulo em $x = L$

---
## 📈 Seção 8: Análise Quantitativa — Fluxo de Calor na Base

In [ ]:
# ============================================================
# CÁLCULO DO FLUXO DE CALOR NA BASE
# ============================================================

def calcular_fluxo_calor(T, dx):
    """
    Calcula o fluxo de calor na base por diferença finita progressiva.

    Parameters
    ----------
    T : ndarray
        Vetor de temperaturas [°C].
    dx : float
        Espaçamento entre nós [m].

    Returns
    -------
    float
        Fluxo de calor na base [W].

    Notes
    -----
    q = -k * A_c * dT/dx|_{x=0} ≈ -k * A_c * (T_1 - T_0) / dx
    """
    dTdx = (T[1] - T[0]) / dx
    return -k * A_c * dTdx

# Calcular para ambas as malhas
q_5nos = calcular_fluxo_calor(T5, dx5)
q_21nos = calcular_fluxo_calor(T21, dx21)

erro_5 = 100.0 * abs(q_5nos - q_analitico) / q_analitico
erro_21 = 100.0 * abs(q_21nos - q_analitico) / q_analitico

print('=' * 60)
print('  FLUXO DE CALOR NA BASE DA ALETA')
print('=' * 60)
print(f'\n📊 Solução analítica : {q_analitico:.4f} W')
print(f'   MDF — 5 nós       : {q_5nos:.4f} W  (erro = {erro_5:.2f}%)')
print(f'   MDF — 21 nós      : {q_21nos:.4f} W  (erro = {erro_21:.2f}%)')
print(f'\n📏 Eficiência da aleta (analítica):')
eta_ana = np.tanh(m * L) / (m * L)
print(f'   η = tanh(mL)/(mL) = {eta_ana:.4f} ({eta_ana*100:.1f}%)')
print(f'\n✅ Fluxo calculado com sucesso para ambas as malhas')

> 💡 **Observação:**
> - O fluxo de calor depende do **gradiente** na base, que é mais sensível à discretização que a própria temperatura
> - A malha grossa (5 nós) apresenta erro de ~3% no fluxo, enquanto a refinada (21 nós) reduz para < 0,1%
> - Isso ilustra que grandezas derivadas (gradientes) exigem malhas mais refinadas que grandezas primárias (temperatura)

---
## 🔍 Seção 9: Estudo de Independência de Malha (Convergência)

In [ ]:
# ============================================================
# ESTUDO DE CONVERGÊNCIA: Refinamento progressivo de malha
# ============================================================

N_list = [5, 11, 21, 41, 81, 161, 321]
q_num = []
erro_q = []
dx_list = []
t_cpu_list = []

for N in N_list:
    t_ini = time.time()
    x, T, dx = resolver_aleta_mdf(N)
    t_cpu_list.append(time.time() - t_ini)
    q = calcular_fluxo_calor(T, dx)
    q_num.append(q)
    erro_q.append(100.0 * abs(q - q_analitico) / q_analitico)
    dx_list.append(dx)

print('=' * 60)
print('  ESTUDO DE INDEPENDÊNCIA DE MALHA')
print('=' * 60)
print(f'\n{"N nós":<10}{"Δx [mm]":<12}{"q [W]":<14}{"Erro [%]":<12}{"CPU [s]":<10}')
print('-' * 58)
for N, dx, q, e, tc in zip(N_list, dx_list, q_num, erro_q, t_cpu_list):
    print(f'{N:<10}{dx*1000:<12.3f}{q:<14.4f}{e:<12.4f}{tc:<10.6f}')

In [ ]:
# ============================================================
# GRÁFICOS DE CONVERGÊNCIA
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- Painel 1: Erro vs Δx (escala log-log) ---
ax1.loglog(dx_list, erro_q, 'o-', color='#d62728', linewidth=2, markersize=8)
# Linha de referência: O(dx^2)
dx_ref = np.array(dx_list)
erro_ref = erro_q[0] * (dx_ref / dx_list[0]) ** 2
ax1.loglog(dx_ref, erro_ref, 'k--', linewidth=1.5, alpha=0.7,
           label='Referência: $\\mathcal{O}(\\Delta x^2)$')
ax1.set_xlabel('Δx [m]', fontsize=12)
ax1.set_ylabel('Erro no fluxo de calor [%]', fontsize=12)
ax1.set_title('Convergência do Erro com Refinamento de Malha',
              fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3, which='both')

# --- Painel 2: Perfis para diferentes malhas ---
ax2.plot(x_ana * 100, T_ana, 'k-', linewidth=2.5, label='Analítica')
cores = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_list)))
for N, cor in zip(N_list, cores):
    x, T, _ = resolver_aleta_mdf(N)
    ax2.plot(x * 100, T, 'o-', color=cor, markersize=4, label=f'N={N}')

ax2.set_xlabel('Posição x [cm]', fontsize=12)
ax2.set_ylabel('Temperatura T [°C]', fontsize=12)
ax2.set_title('Perfis de Temperatura para Diferentes Malhas',
              fontsize=14, fontweight='bold')
ax2.legend(fontsize=9, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - O gráfico log-log confirma a **convergência de segunda ordem**: a inclinação da reta é ≈ 2
> - Ao reduzir $\Delta x$ pela metade, o erro cai por um fator de ~4
> - Para $N \geq 21$, o erro já é inferior a 0,1%, adequado para engenharia
> - O custo computacional cresce linearmente com $N$ (sistema tridiagonal)

In [ ]:
# ============================================================
# TABELA COMPARATIVA DE MALHAS
# ============================================================

import pandas as pd
from IPython.display import display, Markdown

dados_tabela = {
    'Nós (N)': N_list,
    'Δx [mm]': [f'{dx*1000:.2f}' for dx in dx_list],
    'q_num [W]': [f'{q:.4f}' for q in q_num],
    'Erro [%]': [f'{e:.4f}' for e in erro_q],
    'CPU [μs]': [f'{tc*1e6:.1f}' for tc in t_cpu_list]
}

df_conv = pd.DataFrame(dados_tabela)
display(Markdown(df_conv.to_markdown(index=False)))

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Convergência de segunda ordem:** O MDF com diferenças centrais apresenta erro proporcional a $(\Delta x)^2$. Ao reduzir $\Delta x$ pela metade, o erro cai aproximadamente por um fator de 4, confirmando a ordem teórica do esquema.

2. **Sensibilidade do gradiente:** A malha grossa (5 nós) fornece boa estimativa da temperatura na ponta (erro < 1%), mas o fluxo de calor na base — que depende do gradiente — apresenta erro maior (~3%). Grandezas derivadas exigem malhas mais refinadas.

3. **Eficiência da aleta (~76%):** O valor de $mL \approx 1{,}31$ está na faixa intermediária, indicando que a aleta opera com eficiência razoável, mas não máxima. Aletas mais curtas ou com maior $k$ teriam eficiência superior.

### Limitações do Modelo

- **Ponta adiabática:** O modelo despreza a convecção na ponta da aleta. Para aletas curtas ou com $h$ elevado, essa aproximação pode introduzir erro significativo.
- **Propriedades constantes:** O modelo assume $k$ e $h$ uniformes. Na realidade, $k$ pode variar com a temperatura e $h$ varia ao longo da superfície.
- **Regime permanente:** Não considera transientes térmicos (partida, variação de carga).
- **Geometria 1D:** Assume temperatura uniforme na seção transversal (número de Biot pequeno). Para aletas espessas, essa hipótese pode não ser válida.

### Aplicações Práticas

- **Projeto de dissipadores térmicos:** O perfil de temperatura determina a eficiência de cada aleta e o dimensionamento do conjunto.
- **Otimização de geometria:** O estudo de convergência orienta a escolha da malha mínima necessária para uma precisão desejada, reduzindo custo computacional em projetos iterativos.

### Extensões Sugeridas

- Incluir radiação superficial: $q_{\text{rad}} = \varepsilon \sigma (T^4 - T_{\text{sur}}^4)$
- Condutividade térmica dependente da temperatura: $k(T) = k_0[1 + \beta(T - T_\infty)]$
- Seção transversal variável (aleta triangular)
- Método de Volumes Finitos para comparação de conservação de energia

### Conexão com a Teoria

> 📖 **Referência:** A equação da aleta e sua solução analítica são apresentadas em Incropera et al. (2017), Cap. 3, Seções 3.6–3.7. A discretização por diferenças finitas segue a formulação de Gobbi (s.d.), Cap. 2, para conservação de energia em regime permanente. O estudo de convergência é consistente com a análise de erro de truncamento de séries de Taylor.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 10](../notebooks/10_Aletas_Superficies.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 10.7

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>